In [1]:
import os
import gc
import json
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate
# 📦 Imports
import json
from collections import Counter
from pathlib import Path
import os
import evaluate
import pandas as pd
from datasets import Dataset, DatasetDict


# 📊 Evaluation metric
from evaluate import load
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments, EarlyStoppingCallback
)

accuracy_metric = load("accuracy")


/data/literature/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Set this before importing any other libraries that use CUDA
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

print("✅ Using only GPU 3")
print("🧹 Cleared CUDA cache")

# Verify single GPU setup
if torch.cuda.is_available():
    print(f"Available GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f" - GPU {i}: {torch.cuda.get_device_name(i)} (Physical GPU 3)")
else:
    print("❌ No CUDA available!")
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 

✅ Using only GPU 3
🧹 Cleared CUDA cache
Available GPUs: 1
 - GPU 0: NVIDIA RTX A6000 (Physical GPU 3)


In [3]:
def create_poem_dataset(scraper_base_path):
    """
    Creates a dataset of poems and their categories based on the provided JSON structure.
    """
    poets_root_dir = os.path.join(scraper_base_path, "output", "poets")
    dataset = []

    print(f"🔍 Starting scan in: {poets_root_dir}")

    if not os.path.isdir(poets_root_dir):
        print(f"❌ Error: Directory '{poets_root_dir}' not found.")
        return []

    for poet_name in os.listdir(poets_root_dir):
        poet_dir_path = os.path.join(poets_root_dir, poet_name)
        
        if not os.path.isdir(poet_dir_path):
            continue

        metadata_file = os.path.join(poet_dir_path, "metadata.json")

        if os.path.exists(metadata_file):
            try:
                with open(metadata_file, 'r', encoding='utf-8') as f:
                    metadata = json.load(f)

                for poem_info in metadata.get("kavita", []):
                    categories = poem_info.get("categories")
                    path_from_json = poem_info.get("devnagri_path")

                    if categories and path_from_json:
                        absolute_path = os.path.join(scraper_base_path, path_from_json)

                        if os.path.exists(absolute_path):
                            with open(absolute_path, 'r', encoding='utf-8') as poem_file:
                                poem_text = poem_file.read()
                            
                            dataset.append({
                                "text": poem_text,
                                "category": categories[0] # Taking the first category
                            })
                        else:
                            print(f"⚠️ Warning: File not found at '{absolute_path}'")

            except Exception as e:
                print(f"❌ An error occurred while processing '{metadata_file}': {e}")
                
    return dataset

In [4]:
# Set the path to your main project folder
scraper_project_path = "/home/literature/hindwi-scraper"

# This list will now store a list of categories for each poem
poem_data_list_multiple_cat = []

# We re-use the core logic from the create_poem_dataset function here directly
poets_root_dir = os.path.join(scraper_project_path, "output", "poets")
if os.path.isdir(poets_root_dir):
    for poet_name in os.listdir(poets_root_dir):
        poet_dir_path = os.path.join(poets_root_dir, poet_name)
        if not os.path.isdir(poet_dir_path): continue
        metadata_file = os.path.join(poet_dir_path, "metadata.json")
        if os.path.exists(metadata_file):
            with open(metadata_file, 'r', encoding='utf-8') as f:
                metadata = json.load(f)
            for poem_info in metadata.get("kavita", []):
                categories = poem_info.get("categories")
                path_from_json = poem_info.get("devnagri_path")
                if categories and path_from_json:
                    absolute_path = os.path.join(scraper_project_path, path_from_json)
                    if os.path.exists(absolute_path):
                        with open(absolute_path, 'r', encoding='utf-8') as poem_file:
                            poem_text = poem_file.read()
                        # ✨ CHANGE: Store the ENTIRE list of categories
                        poem_data_list_multiple_cat.append({
                            "text": poem_text,
                            "category": categories 
                        })

# Convert the list to a DataFrame
df = pd.DataFrame(poem_data_list_multiple_cat)

# ✨ NEW STEP: Explode the DataFrame. This is the magic part.
# If a row has ["स्त्री", "प्रेम"] in 'category', it becomes two rows.
df_exploded = df.explode('category')

print(f"✅ Loaded {len(df)} poems, which were expanded to {len(df_exploded)} rows to count all categories.")
print("\nNew value counts for categories:")
print(df_exploded['category'].value_counts())

✅ Loaded 20314 poems, which were expanded to 43049 rows to count all categories.

New value counts for categories:
category
स्त्री           1527
प्रेम            1415
चीज़ें            1045
वैश्विक कविता    1035
जीवन             1035
                 ... 
व्यक्ति             1
उपदेश               1
मार्ग               1
अनहद                1
शिष्टाचार           1
Name: count, Length: 603, dtype: int64


In [5]:
# 🎯 Cell 1: Filter categories with sufficient examples
# This prevents overfitting on rare categories

min_examples = 50  # Adjust based on your needs - categories with fewer examples will be excluded
category_counts = df_exploded['category'].value_counts()
frequent_categories = category_counts[category_counts >= min_examples].index

# Filter the dataset to keep only frequent categories
df_filtered = df_exploded[df_exploded['category'].isin(frequent_categories)].copy()

print(f"📊 Dataset after filtering:")
print(f"Original: {len(df_exploded)} examples with {df_exploded['category'].nunique()} categories")
print(f"Filtered: {len(df_filtered)} examples with {len(frequent_categories)} categories")
print(f"Removed {df_exploded['category'].nunique() - len(frequent_categories)} rare categories")

print(f"\nTop 10 categories after filtering:")
print(df_filtered['category'].value_counts().head(10))

print(f"\nBottom 10 categories after filtering:")
print(df_filtered['category'].value_counts().tail(10))

📊 Dataset after filtering:
Original: 43049 examples with 603 categories
Filtered: 38288 examples with 179 categories
Removed 424 rare categories

Top 10 categories after filtering:
category
स्त्री           1527
प्रेम            1415
चीज़ें            1045
जीवन             1035
वैश्विक कविता    1035
संबंध             874
लोक               767
समय               760
आत्म              745
स्मृति            738
Name: count, dtype: int64

Bottom 10 categories after filtering:
category
जर्मन कविता    53
बुद्ध          53
रहस्य          51
पत्नी          51
सर्दी          51
सेक्स          51
दर्शन          51
दरवाज़ा         50
छाया           50
बादल           50
Name: count, dtype: int64


In [6]:
# 🏷️ Cell 2: Create label mappings for model training
# Convert text categories to numeric labels that the model can understand

unique_categories = sorted(df_filtered['category'].unique())
label2id = {label: i for i, label in enumerate(unique_categories)}
id2label = {i: label for label, i in label2id.items()}

print(f"🏷️ Created mappings for {len(unique_categories)} categories")

# Add numeric labels to dataframe
df_filtered['labels'] = df_filtered['category'].map(label2id)

# Show some examples of the mapping
print(f"\nSample label mappings:")
for i, (label, label_id) in enumerate(list(label2id.items())[:10]):
    print(f"{label_id}: {label}")

if len(unique_categories) > 10:
    print(f"... and {len(unique_categories) - 10} more categories")

# Verify no missing labels
missing_labels = df_filtered['labels'].isnull().sum()
print(f"\n✅ Missing labels: {missing_labels} (should be 0)")
print(f"✅ Label range: {df_filtered['labels'].min()} to {df_filtered['labels'].max()}")

🏷️ Created mappings for 179 categories

Sample label mappings:
0: अँग्रेज़ी कविता
1: अँधेरा
2: अकेलापन
3: अवधी कविता
4: अवसाद
5: असमिया कविता
6: अस्तित्व
7: आँख
8: आँसू
9: आकाश
... and 169 more categories

✅ Missing labels: 0 (should be 0)
✅ Label range: 0 to 178


In [7]:
# 📚 Cell 3: Split the data into train/validation/test (80-10-10)
# Using stratified splits to ensure balanced category distribution

# First split: 80% train, 20% temp
train_df, temp_df = train_test_split(
    df_filtered, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_filtered['labels']  # Ensure balanced splits across all categories
)

# Second split: 10% validation, 10% test from the 20% temp
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    random_state=42, 
    stratify=temp_df['labels']
)

print(f"📊 Dataset splits:")
print(f"Train: {len(train_df)} examples ({len(train_df)/len(df_filtered)*100:.1f}%)")
print(f"Validation: {len(val_df)} examples ({len(val_df)/len(df_filtered)*100:.1f}%)")
print(f"Test: {len(test_df)} examples ({len(test_df)/len(df_filtered)*100:.1f}%)")

# Verify stratification worked - all splits should have all categories
print(f"\n✅ Category distribution verification:")
print(f"Train categories: {train_df['category'].nunique()}")
print(f"Val categories: {val_df['category'].nunique()}")
print(f"Test categories: {test_df['category'].nunique()}")
print(f"Expected: {len(unique_categories)} categories in each split")

# Show category distribution across splits for top categories
print(f"\nTop 5 categories across splits:")
top_5_categories = df_filtered['category'].value_counts().head(5).index
for cat in top_5_categories:
    train_count = (train_df['category'] == cat).sum()
    val_count = (val_df['category'] == cat).sum()
    test_count = (test_df['category'] == cat).sum()
    print(f"{cat}: Train={train_count}, Val={val_count}, Test={test_count}")

📊 Dataset splits:
Train: 30630 examples (80.0%)
Validation: 3829 examples (10.0%)
Test: 3829 examples (10.0%)

✅ Category distribution verification:
Train categories: 179
Val categories: 179
Test categories: 179
Expected: 179 categories in each split

Top 5 categories across splits:
स्त्री: Train=1222, Val=153, Test=152
प्रेम: Train=1132, Val=142, Test=141
चीज़ें: Train=836, Val=104, Test=105
जीवन: Train=828, Val=104, Test=103
वैश्विक कविता: Train=828, Val=104, Test=103


In [8]:
# 🤗 Cell 4: Load the IndicBERTv2 tokenizer
model_name = "ai4bharat/IndicBERTv2-MLM-only"

print(f"🔄 Loading tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"✅ Tokenizer loaded successfully!")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Model max length: {tokenizer.model_max_length}")

# Test tokenizer with a sample Hindi text
sample_text = "यह एक हिंदी कविता है।"
tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.encode(sample_text)

print(f"\n🧪 Tokenizer test:")
print(f"Sample text: {sample_text}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"Decoded back: {tokenizer.decode(token_ids)}")

# Check if special tokens are present
print(f"\n🏷️ Special tokens:")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"CLS token: {tokenizer.cls_token} (ID: {tokenizer.cls_token_id})")
print(f"SEP token: {tokenizer.sep_token} (ID: {tokenizer.sep_token_id})")
print(f"UNK token: {tokenizer.unk_token} (ID: {tokenizer.unk_token_id})")

🔄 Loading tokenizer: ai4bharat/IndicBERTv2-MLM-only
✅ Tokenizer loaded successfully!
Vocab size: 250000
Model max length: 1000000000000000019884624838656

🧪 Tokenizer test:
Sample text: यह एक हिंदी कविता है।
Tokens: ['यह', 'एक', 'हिंदी', 'कविता', 'है', '।']
Token IDs: [1, 16227, 15864, 30642, 35433, 15529, 1395, 2]
Decoded back: [CLS] यह एक हिंदी कविता है । [SEP]

🏷️ Special tokens:
PAD token: [PAD] (ID: 3)
CLS token: [CLS] (ID: 1)
SEP token: [SEP] (ID: 2)
UNK token: [UNK] (ID: 0)


In [9]:
# 📏 Cell 5: Analyze poem lengths to determine optimal max_length for tokenization

# Calculate character lengths
poem_lengths_chars = df_filtered['text'].apply(len)

# Calculate token lengths using our tokenizer
print("🔄 Calculating token lengths... (this may take a moment)")
poem_lengths_tokens = df_filtered['text'].apply(lambda x: len(tokenizer.tokenize(x)))

print(f"📊 Poem length analysis:")
print(f"\nCharacter lengths:")
print(f"Mean: {poem_lengths_chars.mean():.0f} chars")
print(f"Median: {poem_lengths_chars.median():.0f} chars")
print(f"95th percentile: {poem_lengths_chars.quantile(0.95):.0f} chars")
print(f"Max: {poem_lengths_chars.max():.0f} chars")

print(f"\nToken lengths:")
print(f"Mean: {poem_lengths_tokens.mean():.0f} tokens")
print(f"Median: {poem_lengths_tokens.median():.0f} tokens")
print(f"95th percentile: {poem_lengths_tokens.quantile(0.95):.0f} tokens")
print(f"Max: {poem_lengths_tokens.max():.0f} tokens")

# Determine optimal max_length
# We want to capture most poems without too much padding
percentiles = [0.8, 0.9, 0.95, 0.99]
print(f"\nToken length percentiles:")
for p in percentiles:
    length = int(poem_lengths_tokens.quantile(p))
    poems_covered = (poem_lengths_tokens <= length).mean() * 100
    print(f"{p*100}%: {length} tokens (covers {poems_covered:.1f}% of poems)")

# Suggest max_length
suggested_max_length = min(512, int(poem_lengths_tokens.quantile(0.95)))
poems_truncated = (poem_lengths_tokens > suggested_max_length).mean() * 100

print(f"\n💡 Suggested max_length: {suggested_max_length}")
print(f"This will truncate {poems_truncated:.1f}% of poems")

# Show some examples of long poems
if poems_truncated > 5:  # If more than 5% will be truncated
    print(f"\n⚠️ Long poem examples (first 200 chars):")
    long_poems = df_filtered[poem_lengths_tokens > suggested_max_length]['text'].head(3)
    for i, poem in enumerate(long_poems, 1):
        print(f"{i}. {poem[:200]}...")
        print(f"   Length: {len(tokenizer.tokenize(poem))} tokens\n")

🔄 Calculating token lengths... (this may take a moment)
📊 Poem length analysis:

Character lengths:
Mean: 685 chars
Median: 504 chars
95th percentile: 1857 chars
Max: 19523 chars

Token lengths:
Mean: 182 tokens
Median: 134 tokens
95th percentile: 492 tokens
Max: 5344 tokens

Token length percentiles:
80.0%: 261 tokens (covers 80.1% of poems)
90.0%: 368 tokens (covers 90.0% of poems)
95.0%: 492 tokens (covers 95.0% of poems)
99.0%: 991 tokens (covers 99.0% of poems)

💡 Suggested max_length: 492
This will truncate 5.0% of poems


In [10]:
# 📝 Cell 6: Create tokenization function and convert to HuggingFace datasets

# Define tokenization function
# Adjust max_length based on the analysis from previous cell
max_length = 512  # You can adjust this based on the previous cell's output

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        padding="max_length",
        max_length=max_length
    )

print(f"🔤 Tokenization settings:")
print(f"Max length: {max_length}")
print(f"Truncation: True")
print(f"Padding: max_length")

# Convert pandas DataFrames to HuggingFace datasets
print(f"\n🔄 Converting to HuggingFace datasets...")

train_dataset = Dataset.from_pandas(train_df[['text', 'labels']])
val_dataset = Dataset.from_pandas(val_df[['text', 'labels']])
test_dataset = Dataset.from_pandas(test_df[['text', 'labels']])

print(f"✅ Datasets created:")
print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")
print(f"Test dataset: {len(test_dataset)} examples")

# Apply tokenization to all datasets
print(f"\n🔄 Tokenizing datasets... (this may take a few minutes)")

train_dataset = train_dataset.map(tokenize_function, batched=True, desc="Tokenizing train")
val_dataset = val_dataset.map(tokenize_function, batched=True, desc="Tokenizing validation")
test_dataset = test_dataset.map(tokenize_function, batched=True, desc="Tokenizing test")

print(f"✅ All datasets tokenized successfully!")

# Show the structure of tokenized data
print(f"\n📋 Tokenized dataset structure:")
print(f"Features: {train_dataset.features}")
print(f"\nSample tokenized example:")
sample = train_dataset[0]
print(f"Input IDs length: {len(sample['input_ids'])}")
print(f"Attention mask length: {len(sample['attention_mask'])}")
print(f"Label: {sample['labels']} (category: {id2label[sample['labels']]})")

# Check for any potential issues
print(f"\n🔍 Data validation:")
sample_batch = train_dataset.select(range(min(100, len(train_dataset))))
input_lengths = [len(ex) for ex in sample_batch['input_ids']]
attention_lengths = [len(ex) for ex in sample_batch['attention_mask']]

print(f"All input_ids same length: {len(set(input_lengths)) == 1}")
print(f"All attention_masks same length: {len(set(attention_lengths)) == 1}")
print(f"Input length: {input_lengths[0] if input_lengths else 'N/A'}")
print(f"Label range: {min(train_dataset['labels'])} to {max(train_dataset['labels'])}")

🔤 Tokenization settings:
Max length: 512
Truncation: True
Padding: max_length

🔄 Converting to HuggingFace datasets...
✅ Datasets created:
Train dataset: 30630 examples
Val dataset: 3829 examples
Test dataset: 3829 examples

🔄 Tokenizing datasets... (this may take a few minutes)


Tokenizing test: 100%|██████████| 3829/3829 [00:00<00:00, 4261.78 examples/s]


✅ All datasets tokenized successfully!

📋 Tokenized dataset structure:
Features: {'text': Value('string'), 'labels': Value('int64'), '__index_level_0__': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}

Sample tokenized example:
Input IDs length: 512
Attention mask length: 512
Label: 59 (category: दर्द)

🔍 Data validation:
All input_ids same length: True
All attention_masks same length: True
Input length: 512
Label range: 0 to 178


In [11]:
# 🤖 Cell 7: Load the IndicBERTv2 model for sequence classification

print(f"🔄 Loading model: {model_name}")
print(f"Number of labels: {len(unique_categories)}")

# Load the model with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(unique_categories),
    id2label=id2label,
    label2id=label2id
)

print(f"✅ Model loaded successfully!")

# Display model information
print(f"\n📊 Model information:")
print(f"Model name: {model_name}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Number of labels: {model.num_labels}")

# Check if model is on correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Available device: {device}")

if torch.cuda.is_available():
    print(f"CUDA devices: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  Device {i}: {torch.cuda.get_device_name(i)}")

# Move model to device (will be handled automatically by Trainer, but good to check)
model = model.to(device)
print(f"✅ Model moved to: {next(model.parameters()).device}")

# Test model with a sample input
print(f"\n🧪 Quick model test:")
sample_text = train_df.iloc[0]['text'][:100] + "..."  # First 100 chars
sample_category = train_df.iloc[0]['category']

# Tokenize sample
sample_inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=max_length)
sample_inputs = {k: v.to(device) for k, v in sample_inputs.items()}

# Forward pass
with torch.no_grad():
    outputs = model(**sample_inputs)
    predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_id = torch.argmax(predictions, dim=-1).item()
    confidence = predictions[0][predicted_id].item()

print(f"Sample text: {sample_text}")
print(f"Actual category: {sample_category}")
print(f"Random prediction: {id2label[predicted_id]} (confidence: {confidence:.3f})")
print(f"Note: This is before training, so prediction is random!")

🔄 Loading model: ai4bharat/IndicBERTv2-MLM-only
Number of labels: 179


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/IndicBERTv2-MLM-only and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded successfully!

📊 Model information:
Model name: ai4bharat/IndicBERTv2-MLM-only
Number of parameters: 278,178,995
Number of labels: 179
Available device: cuda
CUDA devices: 1
  Device 0: NVIDIA RTX A6000
✅ Model moved to: cuda:0

🧪 Quick model test:
Sample text: इतनी निष्ठुर सर्दियाँ
कि शरीर की सारी कोशिकाएँ
सुप्तावस्था में चली जाएँ
मैं जीवित हूँ तुम्हारी याद क...
Actual category: दर्द
Random prediction: एकांत (confidence: 0.007)
Note: This is before training, so prediction is random!


In [12]:
# 📊 Cell 8: Define evaluation metrics for training

# Load additional metrics for comprehensive evaluation
accuracy_metric = load("accuracy")
f1_metric = load("f1")

def compute_metrics(eval_pred):
    """
    Compute accuracy and F1 score
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    # Calculate metrics
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    
    # For multi-class classification, use macro averaging
    f1_results = f1_metric.compute(
        predictions=predictions, 
        references=labels, 
        average="macro"
    )
    
    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1_results["f1"]
    }

print("✅ Evaluation metrics defined:")
print("- Accuracy: Overall classification accuracy")
print("- F1-score: Macro-averaged F1 score across all categories")

# Test the metrics function with dummy data
print(f"\n🧪 Testing metrics function:")
dummy_predictions = np.array([[0.8, 0.1, 0.1], [0.2, 0.7, 0.1], [0.1, 0.2, 0.7]])
dummy_labels = np.array([0, 1, 2])
dummy_eval_pred = (dummy_predictions, dummy_labels)

test_metrics = compute_metrics(dummy_eval_pred)
print("Test metrics (with dummy perfect predictions):")
for metric_name, value in test_metrics.items():
    print(f"  {metric_name}: {value:.4f}")

print(f"\n💡 Note: We're using macro averaging for F1, which:")
print("- Treats all categories equally (good for imbalanced data)")
print("- Averages metrics across categories rather than samples")
print("- Gives equal weight to rare and common categories")

✅ Evaluation metrics defined:
- Accuracy: Overall classification accuracy
- F1-score: Macro-averaged F1 score across all categories

🧪 Testing metrics function:
Test metrics (with dummy perfect predictions):
  accuracy: 1.0000
  f1: 1.0000

💡 Note: We're using macro averaging for F1, which:
- Treats all categories equally (good for imbalanced data)
- Averages metrics across categories rather than samples
- Gives equal weight to rare and common categories


In [13]:
# 🏋️ Cell 9: Configure training arguments

training_args = TrainingArguments(
    # Output and logging
    output_dir="./hindi-poem-classifier",
    logging_dir="./hindi-poem-classifier/logs",
    logging_steps=100,
    
    # Training hyperparameters
    learning_rate=2e-5,
    num_train_epochs=10,
    warmup_steps=500,
    weight_decay=0.01,
    
    # Batch sizes (adjust based on GPU memory)
    per_device_train_batch_size=4,  # Start with 16, reduce if OOM
    per_device_eval_batch_size=4,
    
    # Evaluation and saving
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",  # Need to save to load best model
    save_steps=500,  # Same as eval_steps
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    # Performance optimizations
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    gradient_checkpointing=True,  # Save memory
    fp16=True,  # Mixed precision for faster training
    
    # Storage optimization - only keep the best checkpoint
    save_total_limit=1,  # Keep only 1 checkpoint (the current best)
)

print("🏋️ Training configuration:")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Train batch size: {training_args.per_device_train_batch_size}")
print(f"Eval batch size: {training_args.per_device_eval_batch_size}")
print(f"Warmup steps: {training_args.warmup_steps}")
print(f"Weight decay: {training_args.weight_decay}")
print(f"Mixed precision (fp16): {training_args.fp16}")
print(f"Gradient checkpointing: {training_args.gradient_checkpointing}")

# Calculate approximate training time
total_steps = len(train_dataset) // (training_args.per_device_train_batch_size * torch.cuda.device_count()) * training_args.num_train_epochs
print(f"\n⏱️ Training estimates:")
print(f"Total training steps: ~{total_steps}")
print(f"Evaluation every {training_args.eval_steps} steps")
print(f"No intermediate checkpoints saved (saves storage space)")

print(f"\n💾 Output directory: {training_args.output_dir}")
print(f"📊 Logs directory: {training_args.logging_dir}")

print(f"\n💡 Notes:")
print("- If you get OOM (Out of Memory) errors, reduce batch_size to 8 or 4")
print("- Training will automatically use all available GPUs (3,4,5)")
print("- Only 1 checkpoint kept at a time (the current best based on F1 score)")
print("- Mixed precision (fp16) speeds up training on modern GPUs")
print("- save_total_limit=1 keeps storage usage minimal")

🏋️ Training configuration:
Learning rate: 2e-05
Epochs: 10
Train batch size: 4
Eval batch size: 4
Warmup steps: 500
Weight decay: 0.01
Mixed precision (fp16): True
Gradient checkpointing: True

⏱️ Training estimates:
Total training steps: ~76570
Evaluation every 500 steps
No intermediate checkpoints saved (saves storage space)

💾 Output directory: ./hindi-poem-classifier
📊 Logs directory: ./hindi-poem-classifier/logs

💡 Notes:
- If you get OOM (Out of Memory) errors, reduce batch_size to 8 or 4
- Training will automatically use all available GPUs (3,4,5)
- Only 1 checkpoint kept at a time (the current best based on F1 score)
- Mixed precision (fp16) speeds up training on modern GPUs
- save_total_limit=1 keeps storage usage minimal


In [14]:
# 🎯 Cell 10: Initialize the Trainer with early stopping

# Initialize trainer with early stopping callback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("🎯 Trainer initialized successfully!")

# Display trainer configuration summary
print(f"\n📋 Trainer Configuration Summary:")
print(f"Model: {model_name}")
print(f"Training examples: {len(train_dataset):,}")
print(f"Validation examples: {len(val_dataset):,}")
print(f"Test examples: {len(test_dataset):,}")
print(f"Number of categories: {len(unique_categories)}")
print(f"Device count: {torch.cuda.device_count()}")

# Training parameters summary
print(f"\n🏋️ Training Parameters:")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Train batch size per device: {training_args.per_device_train_batch_size}")
print(f"Total effective batch size: {training_args.per_device_train_batch_size * torch.cuda.device_count()}")
print(f"Warmup steps: {training_args.warmup_steps}")
print(f"Weight decay: {training_args.weight_decay}")
print(f"Early stopping patience: 3 evaluations without improvement")

# Memory and performance settings
print(f"\n⚡ Performance Settings:")
print(f"Mixed precision (fp16): {training_args.fp16}")
print(f"Gradient checkpointing: {training_args.gradient_checkpointing}")
print(f"DataLoader workers: {training_args.dataloader_num_workers}")
print(f"Pin memory: {training_args.dataloader_pin_memory}")

# Final checks before training
print(f"\n✅ Pre-training Checks:")
print(f"Model on correct device: {next(model.parameters()).device}")
print(f"Training dataset ready: {len(train_dataset)} examples")
print(f"Validation dataset ready: {len(val_dataset)} examples")
print(f"Tokenizer loaded: {tokenizer is not None}")
print(f"Compute metrics function: {compute_metrics is not None}")

print(f"\n🚀 Ready to start training!")
print(f"💡 Tip: Training will take several hours. You can monitor progress in real-time.")

🎯 Trainer initialized successfully!

📋 Trainer Configuration Summary:
Model: ai4bharat/IndicBERTv2-MLM-only
Training examples: 30,630
Validation examples: 3,829
Test examples: 3,829
Number of categories: 179
Device count: 1

🏋️ Training Parameters:
Epochs: 10
Learning rate: 2e-05
Train batch size per device: 4
Total effective batch size: 4
Warmup steps: 500
Weight decay: 0.01
Early stopping patience: 3 evaluations without improvement

⚡ Performance Settings:
Mixed precision (fp16): True
Gradient checkpointing: True
DataLoader workers: 4
Pin memory: True

✅ Pre-training Checks:
Model on correct device: cuda:0
Training dataset ready: 30630 examples
Validation dataset ready: 3829 examples
Tokenizer loaded: True
Compute metrics function: True

🚀 Ready to start training!
💡 Tip: Training will take several hours. You can monitor progress in real-time.


/tmp/ipykernel_4183295/1015637651.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [15]:
training_args_single_gpu = TrainingArguments(
    output_dir="./hindi-poem-classifier",
    logging_dir="./hindi-poem-classifier/logs",
    logging_steps=100,
    
    # Training parameters
    learning_rate=2e-5,
    num_train_epochs=10,
    warmup_steps=300,
    weight_decay=0.01,
    
    # Single GPU batch sizes - can be larger since no multi-GPU overhead
    per_device_train_batch_size=8,  # Start with 8, can increase if no OOM
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,  # Effective batch = 8 * 2 = 16
    
    # Evaluation settings
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    # Memory optimizations
    dataloader_num_workers=2,  # Can use more workers with single GPU
    dataloader_pin_memory=True,
    gradient_checkpointing=True,
    fp16=True,
    remove_unused_columns=True,
    
    # Storage
    save_total_limit=1,
)

print("🏋️ Single GPU Training Settings:")
print(f"Batch size per device: {training_args_single_gpu.per_device_train_batch_size}")
print(f"Gradient accumulation: {training_args_single_gpu.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args_single_gpu.per_device_train_batch_size * training_args_single_gpu.gradient_accumulation_steps}")
print(f"Epochs: {training_args_single_gpu.num_train_epochs}")
print(f"Workers: {training_args_single_gpu.dataloader_num_workers}")

# Use full dataset since we have more memory available per GPU
print(f"\n📊 Dataset sizes:")
print(f"Train: {len(train_dataset):,} examples")
print(f"Val: {len(val_dataset):,} examples")

# Create single GPU trainer
trainer_single = Trainer(
    model=model,
    args=training_args_single_gpu,
    train_dataset=train_dataset,  # Full dataset
    eval_dataset=val_dataset,     # Full dataset
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("✅ Single GPU trainer created!")
print("\n💡 Benefits of single GPU:")
print("- No multi-GPU communication overhead")
print("- More memory available per batch")
print("- Simpler setup, fewer potential issues")
print("- Can use larger batch sizes")
print(f"\n🚀 Ready to train on single GPU 3!")

🏋️ Single GPU Training Settings:
Batch size per device: 8
Gradient accumulation: 2
Effective batch size: 16
Epochs: 10
Workers: 2

📊 Dataset sizes:
Train: 30,630 examples
Val: 3,829 examples
✅ Single GPU trainer created!

💡 Benefits of single GPU:
- No multi-GPU communication overhead
- More memory available per batch
- Simpler setup, fewer potential issues
- Can use larger batch sizes

🚀 Ready to train on single GPU 3!


In [16]:
# 🚀 Single GPU Training

import time

print("🔥 Starting single GPU training...")
print(f"Using: 1 GPU, batch_size={training_args_single_gpu.per_device_train_batch_size}")
print("=" * 50)

start_time = time.time()

try:
    # Start single GPU training
    training_results = trainer_single.train()
    
    end_time = time.time()
    training_duration = end_time - start_time
    
    print("=" * 50)
    print("✅ Single GPU training completed successfully!")
    print(f"⏱️ Total training time: {training_duration/3600:.2f} hours ({training_duration/60:.1f} minutes)")
    print(f"📊 Final step: {training_results.global_step}")
    
    # Check final memory usage
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1e9
        cached = torch.cuda.memory_reserved(0) / 1e9
        print(f"Final GPU memory: {allocated:.1f}GB allocated, {cached:.1f}GB cached")
    
    print(f"\n🎉 Training successful with single GPU!")
    
except torch.cuda.OutOfMemoryError as e:
    print(f"❌ OOM Error even with single GPU!")
    print("📉 Let's try reducing batch size...")
    
    # Automatically retry with smaller batch size
    print(f"\n🔄 Retrying with batch_size=4...")
    
    # Update training args with smaller batch
    training_args_single_gpu.per_device_train_batch_size = 4
    training_args_single_gpu.per_device_eval_batch_size = 4
    training_args_single_gpu.gradient_accumulation_steps = 4  # Keep effective batch = 16
    
    # Create new trainer with smaller batch
    trainer_small_batch = Trainer(
        model=model,
        args=training_args_single_gpu,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )
    
    print("🔄 Trying again with batch_size=4...")
    
    try:
        training_results = trainer_small_batch.train()
        print("✅ Success with reduced batch size!")
        
    except Exception as retry_error:
        print(f"❌ Still failed: {str(retry_error)[:100]}...")
        print("💡 Try batch_size=2 or reduce max_length to 256")
        
except Exception as e:
    print(f"❌ Training failed: {type(e).__name__}")
    print(f"Error: {str(e)[:200]}...")
    raise e

print(f"\n💡 If successful, next steps:")
print("- Evaluate on test set")
print("- Save final model")
print("- Test inference on new poems")

🔥 Starting single GPU training...
Using: 1 GPU, batch_size=8


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss,Validation Loss,Accuracy,F1
500,4.852700,4.809706,0.041003,0.000987
1000,4.816700,4.796022,0.044398,0.001263
1500,4.798300,4.798272,0.041264,0.001075
2000,4.609000,4.524562,0.093497,0.010993
2500,4.346100,4.293152,0.139984,0.032633
3000,4.194100,4.167582,0.152259,0.043944
3500,4.056800,4.020372,0.170802,0.055646
4000,3.892000,3.949884,0.174980,0.065339
4500,3.844700,3.914094,0.191434,0.082374
5000,3.781900,3.865467,0.194045,0.085992


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

✅ Single GPU training completed successfully!
⏱️ Total training time: 0.84 hours (50.6 minutes)
📊 Final step: 14500
Final GPU memory: 3.4GB allocated, 6.1GB cached

🎉 Training successful with single GPU!

💡 If successful, next steps:
- Evaluate on test set
- Save final model
- Test inference on new poems
